# Stage 4: CrTe2 heterostructures
Relaxed CrTe2 (Stage 1) is lattice-matched onto the selected relaxed substrate slabs (Stage 3) with HeteroBuilder and ZSL. The film takes all of the strain, and the in-plane registry is whatever the interface builder produces.

Each slab keeps the lowest-strain distinct twist, flagged `primary`, which is the only structure that gets the Stage 5 strain series. Strains within 0.1% count as tied, and the smaller cell wins. Each kept match is written at the start gap, at every scan gap, and at the large-separation reference.

**Proof-of-concept choices to review with the experimentalists:** twist orientation, terminations, and facets and phases, which may depend on the growth method. Every distinct candidate is listed in `matches.csv`, and every slab that was not used appears in the manifest as `skipped` with a reason.

**Visual check:** supercells, registry, twist angle, interfacial gap, and matching strain.

In [ ]:
config_path = '../config/pipeline.yaml'
stage = '04_heterostructures'
stage_dir = '../../stages/04_heterostructures'

In [ ]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from IPython.display import display
from pymatgen.core import Composition, Structure
from ornl_lotf import hetero, viz
from ornl_lotf.manifest import read_manifest, write_manifest
warnings.filterwarnings('ignore')
config = yaml.safe_load(Path(config_path).read_text())
out = Path(stage_dir); out.mkdir(parents=True, exist_ok=True)
slabs = read_manifest(out.parent / '03_slab_relax'); bulks = read_manifest(out.parent / '01_bulk_relax')
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_rows', 300)
print('matching:', config['matching']); print('film layers:', config['heterostructures']['film_layers'])

## Build

In [ ]:
records, rows, all_matches = hetero.build_heterostructures(config, slabs, bulks, out)
write_manifest(out, records)
table = pd.DataFrame(rows); table.to_csv(out / 'heterostructures.csv', index=False)
matches = pd.DataFrame(all_matches); matches.to_csv(out / 'matches.csv', index=False)
print(len([r for r in records if r.status == 'done']), 'structures written')
display(table[table.role == 'start'].drop(columns=['role']))

## Slabs not carried forward
These are kept in the manifest so that the full set of slabs that was considered stays visible.

In [ ]:
display(pd.DataFrame([{'slab': r.parent, 'reason': r.extra['skip_reason']} for r in records if r.status == 'skipped']))

## All distinct matches (for the twist discussion)
One row per slab, film thickness, folded twist and cell size, within the strain and atom limits. The first row per slab is the lowest-strain match.

In [ ]:
display(matches)

## Interface top views (start gap)
Filled markers are the top substrate plane and open markers are the bottom film plane; the black outline is the supercell.

In [ ]:
start = [r for r in records if r.status == 'done' and r.extra['gap_role'] == 'start']
film_el = {el.symbol for el in Composition(config['heterostructures']['film']).elements}
titles = [f"{r.polymorph} {''.join(map(str, r.facet))} {r.termination}\n{r.film_layers}L | twist {r.twist_angle}° | strain {100*r.matching_strain['max_abs']:.2f}% | {r.extra['natoms']} at" for r in start]
fig = viz.interface_top_views([Structure.from_file(r.path) for r in start], titles, film_el)
fig.savefig(out / 'interface_top_views.png', dpi=130); plt.show()

## Side views (start gap, z up)

In [ ]:
fig = viz.side_views([Structure.from_file(r.path) for r in start], titles, cols=len(start), supercell=(1, 1, 1), frozen=False)
fig.savefig(out / 'heterostructure_sideviews.png', dpi=130); plt.show()

## Gap series (primary match per slab)

In [ ]:
display(table.pivot_table(index='slab', columns='gap', values='natoms', aggfunc='count', fill_value=0))

## 3D views (start gap)

In [ ]:
viz.show_grid([Structure.from_file(r.path) for r in start], [r.id.removeprefix('het-') for r in start], cols=3, supercell=(1, 1, 1), panel=330)